<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 6 → Mineração de Dados em IoT</div>

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Gerando os Dados e Criando o DataFrame</b></span>

In [ ]:
# GERANDO OS DADOS
def gerar_dados():
    np.random.seed(7)

    temperatura = np.r_[
        np.random.normal(28, 1.2, 60),
        np.random.normal(36, 1.4, 60)
    ]

    vibracao = np.r_[
        np.random.normal(2.2, 0.35, 60),
        np.random.normal(4.8, 0.45, 60)
    ]

    grupo = np.r_[np.zeros(60), np.ones(60)]

    return temperatura, vibracao, grupo

# CRIANDO OS GRÁFICOS E OS PADRÕES 
def plotar_padroes(temperatura, vibracao, grupo):
    plt.figure(figsize=(8, 5))

    for valor, nome in [(0, "Operação normal"), (1, "Alta carga")]:
        plt.scatter(
            temperatura[grupo == valor],
            vibracao[grupo == valor],
            alpha=0.75,
            label=nome
        )

    plt.title("Mineração de Dados IoT: Identificação de Padrões")
    plt.xlabel("Temperatura (°C)")
    plt.ylabel("Vibração")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


temperatura, vibracao, grupo = gerar_dados()
plotar_padroes(temperatura, vibracao, grupo)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
def gerar_dados(n=60):
    np.random.seed(42)

    return pd.concat([
        pd.DataFrame({
            "temperatura": np.random.normal(28, 1.0, n),
            "vibracao": np.random.normal(2.0, 0.25, n)
        }),
        pd.DataFrame({
            "temperatura": np.random.normal(33, 1.0, n),
            "vibracao": np.random.normal(3.5, 0.30, n)
        }),
        pd.DataFrame({
            "temperatura": np.random.normal(38, 1.1, n),
            "vibracao": np.random.normal(5.0, 0.35, n)
        })
    ], ignore_index=True)


def aplicar_kmeans(df):
    X = df[["temperatura", "vibracao"]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    modelo = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["cluster"] = modelo.fit_predict(X_scaled)

    centros = scaler.inverse_transform(modelo.cluster_centers_)

    ordem = np.argsort(centros[:, 0])
    nomes = {
        ordem[0]: "Operação normal",
        ordem[1]: "Carga média",
        ordem[2]: "Alta carga"
    }

    df["regime"] = df["cluster"].map(nomes)

    return centros


def plotar_clusters(df, centros):
    plt.figure(figsize=(8, 5))

    for regime in ["Operação normal", "Carga média", "Alta carga"]:
        parte = df[df["regime"] == regime]

        plt.scatter(
            parte["temperatura"],
            parte["vibracao"],
            alpha=0.75,
            label=regime
        )

    plt.scatter(
        centros[:, 0],
        centros[:, 1],
        marker="X",
        s=220,
        edgecolor="black",
        label="Centros"
    )

    plt.title("Clusterização de Sensores IoT com K-Means")
    plt.xlabel("Temperatura (°C)")
    plt.ylabel("Vibração")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


df = gerar_dados()
centros = aplicar_kmeans(df)
plotar_clusters(df, centros)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
# GERANDO DADOS
def gerar_dados():
    np.random.seed(7)

    tempo = pd.date_range(
        "2026-09-26 08:00",
        periods=100,
        freq="1min"
    )

    sinal = (
        30
        + 2 * np.sin(np.arange(100) / 8)
        + np.random.normal(0, 0.5, 100)
    )

    sinal[35] += 7
    sinal[70] -= 6

    return tempo, sinal

# GRÁFICO DE ANOMALIAS
def plotar_anomalias(tempo, sinal):
    idx = [35, 70]

    plt.figure(figsize=(10, 4))

    plt.plot(
        tempo,
        sinal,
        linewidth=2,
        label="Sensor"
    )

    plt.scatter(
        tempo[idx],
        sinal[idx],
        s=100,
        label="Possíveis anomalias"
    )

    plt.title("Mineração de Dados IoT: Detecção de Anomalias")
    plt.xlabel("Tempo")
    plt.ylabel("Valor do sensor")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

tempo, sinal = gerar_dados()
plotar_anomalias(tempo, sinal)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Gerando os Dados e Criando o DataFrame</b></span>

In [ ]:
# GERANDO OS DADOS
df_ex = pd.DataFrame({
    "setor": ["Hortifruti", "Padaria", "Bebidas", "Limpeza", "Açougue"],
    "visitas": [420, 390, 310, 210, 280],
    "compras": [250, 260, 165, 75, 190],
    "tempo_medio_min": [9.7, 8.0, 5.2, 3.5, 6.6]
})

df_ex ["taxa_conversao"] = (df_ex ["compras"] / df_ex ["visitas"] * 100).round(1)

print(df_ex)


> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Criando Função</b></span>

In [ ]:
# FUNÇÃO PARA GRÁFICOS DE BARRAS
def grafico_barra(coluna, titulo, ylabel):
    plt.figure(figsize=(9, 5))
    plt.bar(df_ex["setor"], df_ex[coluna])
    plt.title(titulo)
    plt.xlabel("Setor")
    plt.ylabel(ylabel)
    plt.xticks(rotation=20)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


# RESULTADOS
print("\nTaxa de conversão por setor:")

for _, linha in df_ex.iterrows():
    print(f"{linha['setor']}: {linha['taxa_conversao']}%")

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Criando e Visualizando os Gráficos</b></span>

In [ ]:
# GRÁFICOS
grafico_barra(
    "visitas",
    "Quantidade de visitas por setor",
    "Número de visitas"
)

grafico_barra(
    "taxa_conversao",
    "Taxa de conversão por setor",
    "Compras / Visitas (%)"
)


# RELAÇÕES ----> TEMPO x CONVERSÃO
plt.figure(figsize=(8, 5))

plt.scatter(
    df_ex["tempo_medio_min"],
    df_ex["taxa_conversao"],
    s=100
)

for _, linha in df_ex.iterrows():
    plt.text(
        linha["tempo_medio_min"] + 0.1,
        linha["taxa_conversao"] + 0.5,
        linha["setor"]
    )

plt.title("Tempo médio de permanência x taxa de conversão")
plt.xlabel("Tempo médio de permanência (min)")
plt.ylabel("Taxa de conversão (%)")
plt.grid(alpha=0.3)

plt.show()